In [1]:

from langchain_ollama import ChatOllama


In [9]:


model = ChatOllama(
    model="llama3.2",
    temperature=0
)

In [ ]:
import requests, pathlib

url = "https://storage.googleapis.com/benchmarks-artifacts/chinook/Chinook.db"
local_path = pathlib.Path("Chinook.db") 

if local_path.exists():
    print(f"{local_path} already exists, skipping download.")
else:
    response = requests.get(url)
    if response.status_code == 200:
        local_path.write_bytes(response.content)
        print(f"File downloaded and saved as {local_path}")
    else:
        print(f"Failed to download the file. Status code: {response.status_code}")

Chinook.db already exists, skipping download.


In [1]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

print(f"Dialect: {db.dialect}")
print(f"Available tables: {db.get_usable_table_names()}")
print(f'Sample output: {db.run("SELECT * FROM Artist LIMIT 5;")}')

Dialect: sqlite
Available tables: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']
Sample output: [(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains')]


In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit   

toolkit = SQLDatabaseToolkit(db=db, llm=model)

tools = toolkit.get_tools()

for tool in tools:
    print(f"{tool.name}: {tool.description}\n")

sql_db_query: Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.

sql_db_schema: Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3

sql_db_list_tables: Input is an empty string, output is a comma-separated list of tables in the database.

sql_db_query_checker: Use this tool to double check if your query is correct before executing it. Always use this tool before executing a query with sql_db_query!



In [16]:
system_prompt = """
You are a SQL expert.

STRICT RULES:
1. You MUST call sql_db_list_tables before calling sql_db_schema.
2. You MUST use exact table names with correct capitalization.
3. This database is SQLite.
4. SQLite does NOT support information_schema.
5. NEVER guess table names.
6. ALWAYS use sql_db_query_checker before sql_db_query.
""".format(
    dialect=db.dialect,
    top_k=5,
)

In [ ]:
from langchain.agents import create_tool_calling_agent, AgentExecutor


agent = create_tool_calling_agent(llm = model, tools = tools)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)


ImportError: cannot import name 'AgentExecutor' from 'langchain.agents' (c:\Uddipto\coding\AssetOpsBench\langchain-academy\lc-academy-env\Lib\site-packages\langchain\agents\__init__.py)

In [18]:
question = "Which genre on average has the longest tracks?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Which genre on average has the longest tracks?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (caea4c66-cad9-49fb-85c0-e54a4dd90896)
 Call ID: caea4c66-cad9-49fb-85c0-e54a4dd90896
  Args:
    tool_input: 
  sql_db_query_checker (707879ee-b5bb-439b-bfbb-96061c60c127)
 Call ID: 707879ee-b5bb-439b-bfbb-96061c60c127
  Args:
    query: SELECT genre, AVG(track_length) FROM music_tracks GROUP BY genre
  sql_db_schema (68e7e5e2-0cb1-4078-bc3d-b6a0946bcc47)
 Call ID: 68e7e5e2-0cb1-4078-bc3d-b6a0946bcc47
  Args:
    table_names: music_tracks,music_genres
================================= Tool Message =================================
Name: sql_db_schema

Error: table_names {'music_tracks', 'music_genres'} not found in database
================================== Ai Message ==================================

It seems like the tool was unable to find 